# 03 — Model: what predicts a data-center ZCTA?

Design decisions, all forced by notebook 02:

1. **Spatially-blocked CV.** KMeans on projected centroids into 25 blocks, grouped
   into 8 folds. A random split leaks across clustered positives.
2. **Average precision, not ROC AUC.** At a 1.6% base rate ROC AUC flatters
   everything; AP against the base rate ("lift") is the honest number.
3. **Nested feature groups.** The only question worth answering is whether the
   PHYSICAL indicators explain anything that plain urbanization does not — so we
   fit URBAN alone, then add POWER, then add PHYSICAL, and read the increments.
4. **`fiber_score` / `fac_dist_m` excluded** from the honest model. Included once,
   at the bottom, only to show how large the circularity is.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score, roc_auc_score
import dcmodel as D

df = pd.read_csv(D.CACHE / "model_table.csv", dtype={"zcta": str})
print(f"{len(df):,} ZCTAs")
for name, cols in [("URBAN", D.URBAN), ("POWER", D.POWER),
                   ("PHYSICAL", D.PHYSICAL), ("LEAKY", D.LEAKY)]:
    print(f"  {name:<9} {len(cols):>2} features")

## 1. Spatial blocks

In [ ]:
blocks = D.spatial_blocks(df, n_blocks=25)
fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter(df["lon"], df["lat"], s=0.7, c=blocks, cmap="tab20", linewidths=0)
ax.set_xlim(-125, -66); ax.set_ylim(24, 50); ax.set_aspect(1.3)
ax.set_xticks([]); ax.set_yticks([])
ax.set_title("25 spatial blocks -> 8 CV folds (no fold shares a metro with another)")
plt.tight_layout(); plt.show()

## 2. Nested comparison

`lift` = average precision divided by the base rate. Lift of 1.0 means the model
is no better than guessing at random.

In [ ]:
groups = {
    "URBAN only":           D.URBAN,
    "PHYSICAL only":        D.PHYSICAL,
    "POWER only":           D.POWER,
    "URBAN + POWER":        D.URBAN + D.POWER,
    "URBAN + POWER + PHYS": D.URBAN + D.POWER + D.PHYSICAL,
    "  + leaky fiber":      D.URBAN + D.POWER + D.PHYSICAL + D.LEAKY,
}

for label in ["has_dc_any", "has_dc_10000"]:
    res, base = D.eval_groups(df, label, groups)
    print(f"--- {label}   positives={int(df[label].sum())}   base rate={100*base:.2f}% ---")
    print(res.to_string(index=False, float_format=lambda x: f"{x:8.4f}"))
    print()

### What this says

- **PHYSICAL alone barely works** (lift ~1.6x, ROC 0.64). The 23 physical
  suitability indicators, on their own, hardly predict where data centers are.
- **URBAN alone does most of the work** (lift ~4.4x).
- **POWER adds almost nothing on top of URBAN** — grid access is already implied by
  being in a city.
- **PHYSICAL adds a real but modest increment** on top of URBAN+POWER: 4.45 -> 4.88
  for any DC (+10%), and 5.23 -> 7.33 for >= 1 ha facilities (+40%).
- **The physical increment grows with facility size.** That is the most interesting
  result here, and it is the one that supports the product: bigger campuses are
  more constrained by land, terrain and water, exactly as Merascope assumes.
- **Adding fiber triples the lift — but most of that is identity overlap**, not skill.
  Quantified in the next section.

## 2b. How much of the fiber gain is circular?

`has_dc_noncolo` keeps only positives at least 1 km from any PeeringDB facility —
i.e. it strips the data centers where the feature and the label are the same
building. If fiber's advantage survives that, it is real signal.

In [ ]:
import geopandas as gpd

geom = gpd.read_file(D.CACHE / "zcta_geom.geojson")
dc   = D.load_datacenters()
df, dist = D.attach_colocation(df, geom, dc)

print(f"within 100 m of a PeeringDB facility: {(dist < 100).sum()} / {len(dist)} "
      f"({100*(dist < 100).mean():.1f}%)")
print(f"positives, all data centers : {df['has_dc_any'].sum()}")
print(f"positives, non-colocated    : {df['has_dc_noncolo'].sum()}")
print()

pair = {"URBAN+POWER+PHYS": D.URBAN + D.POWER + D.PHYSICAL,
        "  + fiber":        D.URBAN + D.POWER + D.PHYSICAL + D.COLO}
for label in ["has_dc_any", "has_dc_noncolo"]:
    res, base = D.eval_groups(df, label, pair)
    mult = res["lift"].iloc[1] / res["lift"].iloc[0]
    print(f"--- {label}  positives={int(df[label].sum())}  base={100*base:.2f}% ---")
    print(res.to_string(index=False, float_format=lambda x: f"{x:8.4f}"))
    print(f"    fiber multiplier: {mult:.2f}x\n")

Fiber multiplies lift by **3.07x** against the full label but only **1.33x** against
the non-colocated one. So roughly two-thirds of its apparent power was the label
restating itself, and one-third is genuine interconnection signal.

**Practical rule: keep `fiber_score` in the model, but report it against
`has_dc_noncolo`.** Dropping it entirely discards real predictive power; quoting the
full-label number overstates the model by a factor of three.

Note too that URBAN+POWER+PHYS scores *higher* on the non-colocated label
(5.63x vs 4.88x). Data centers away from carrier hotels are the more greenfield
ones, and they are exactly the ones the physical indicators predict best — the same
gradient as the facility-size result.

## 3. What the honest model keys on

In [ ]:
label = "has_dc_any"
cols  = [c for c in D.URBAN + D.POWER + D.PHYSICAL if c in df.columns]
X, y  = D.feature_matrix(df, cols).to_numpy(), df[label].to_numpy()

clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06,
                                     max_leaf_nodes=15, l2_regularization=1.0,
                                     random_state=0)
p = cross_val_predict(clf, X, y, cv=GroupKFold(n_splits=8), groups=blocks,
                      method="predict_proba")[:, 1]
df["p_dc"] = p
print(f"spatially-blocked AP = {average_precision_score(y, p):.4f}"
      f"   lift = {average_precision_score(y, p)/y.mean():.2f}x"
      f"   ROC AUC = {roc_auc_score(y, p):.3f}")

clf.fit(X, y)
pi  = permutation_importance(clf, X, y, n_repeats=5, random_state=0,
                             scoring="average_precision", n_jobs=4)
imp = pd.DataFrame({"feature": cols, "imp": pi.importances_mean}) \
        .sort_values("imp", ascending=False)
print()
print(imp.head(15).to_string(index=False, float_format=lambda x: f"{x:8.4f}"))

In [ ]:
t = imp.head(14).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(t["feature"], t["imp"], color="#4c72b0")
ax.set_xlabel("permutation importance (drop in average precision)")
ax.set_title("What the model actually keys on")
plt.tight_layout(); plt.show()

## 4. The product output: high-probability ZCTAs with no mapped data center

This is the part that would ship — ZCTAs the model thinks look like data-center
ground but where OSM shows nothing yet.

**Read these with care.** A high score here means one of three things, and the
model cannot tell them apart:
1. a genuine untargeted site,
2. a site that already has a data center OSM simply has not mapped,
3. a dense-urban ZCTA the model likes for the wrong (urbanization) reason.

In [ ]:
top = df[df[label] == 0].nlargest(30, "p_dc")[
    ["zcta", "state", "p_dc", "area_km2", "pop_exposure_score_nat", "tx_score_nat"]]
print(top.to_string(index=False, float_format=lambda x: f"{x:8.3f}"))
print()
print("State distribution of the top 300:")
print(df[df[label] == 0].nlargest(300, "p_dc")["state"].value_counts().head(12).to_string())

In [ ]:
df.sort_values("p_dc", ascending=False)[
    ["zcta", "state", "lat", "lon", "p_dc", "dc_count", "dc_area_max"]
].to_csv(D.CACHE / "zcta_predictions.csv", index=False)
print("wrote cache/zcta_predictions.csv")

## Limitations — read before quoting any of this

- **This is retrodiction, not prediction.** The label is *where data centers already
  are*. Nothing here has a time dimension, so the model cannot say where the next
  one goes; it says what existing sites have in common. Only 29 of 1,809 OSM
  features carry a `start_date`, so a temporal holdout is not available from OSM.
- **The label is OSM presence, which is coverage-biased.** Under-mapping in rural
  areas and over-mapping of small urban colo suites both push the model toward
  "cities". Several of the top no-DC predictions are Manhattan ZCTAs, which almost
  certainly *do* contain colocation facilities that OSM has not tagged.
- **A footprint is not a capacity.** 1 ha of building is a proxy for scale, not for
  megawatts. Two campuses with the same footprint can differ tenfold in load.
- **Features are state-relative underneath.** The `*_nat` columns are a second
  percentile stretch applied on top of per-state scores, not a re-ranking from raw
  physical units, so cross-state comparisons carry some residual state effect.